# CoarsePCA – Grob-Registrierung auf realen Daten

Erste Erprobung der PCA-basierten Grob-Registrierung (AP2.2) auf einem realen CMM-Scan gegen das zugehörige CAD-Ideal.

**Inputs:**
- CAD-Ideal: PLY-Punktwolke aus der CAD-Konvertierungs-API
- CMM-Scan: segmentiertes `WeldVolumeModel` aus AP2.1 (Preprocessing + RANSAC-Segmentierung abgeschlossen)

**Untersuchungsfragen:**
1. Lage der beiden Punktwolken zueinander vor der Registrierung – identisches oder unterschiedliches Koordinatensystem
2. Qualität der durch CoarsePCA erzielten Ausrichtung, gemessen am mittleren Nächste-Nachbar-Abstand
3. Restfehler nach CoarsePCA als Eingangsgröße für die anschließende Fein-Registrierung (ICP)

## 1 – Setup

In [ ]:
import copy
import logging
from pathlib import Path

import numpy as np
import open3d as o3d
import plotly.graph_objects as go

from schweiss_ki.core.data_structures import WeldVolumeModel
from schweiss_ki.subtraction.registration import CoarsePCA, RegistrationPipeline
from schweiss_ki.subtraction.registration import ICPFine

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')


def plot_clouds(traces, title, height=600):
    fig = go.Figure(data=traces)
    fig.update_layout(
        title=title, height=height,
        scene=dict(aspectmode='data',
                   xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)'),
    )
    return fig


def make_scatter(pts, name, color, size=1.2, opacity=0.5, max_pts=80_000):
    if len(pts) > max_pts:
        idx = np.random.default_rng(0).choice(len(pts), max_pts, replace=False)
        pts = pts[idx]
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers',
        marker=dict(size=size, color=color, opacity=opacity),
        name=name,
    )

## 2 – Daten laden

Lädt die CAD-Punktwolke aus dem PLY-File sowie den segmentierten Scan über `WeldVolumeModel.load()`. Die ausgegebenen Bounding-Box-Maße und Schwerpunkt-Koordinaten erlauben einen ersten quantitativen Vergleich der beiden Koordinatensysteme.

In [ ]:
CAD_PLY = Path('../data/processed/test_files/Baugruppe_Beispielteil_V-Naht_1.5mm_Spalt/pointcloud.ply')
SCAN_MODEL_DIR = Path('../data/processed/test_files/SCHWEIßSPALT_1,0_auf_2,5')

cad_pcd = o3d.io.read_point_cloud(str(CAD_PLY))
print(f'CAD: {len(cad_pcd.points):,} Punkte')
cad_bbox = cad_pcd.get_axis_aligned_bounding_box()
print(f'  BBox-Extent: {cad_bbox.get_extent()}')
print(f'  Schwerpunkt: {cad_bbox.get_center()}')

scan_model = WeldVolumeModel.load(SCAN_MODEL_DIR)
scan_pcd = scan_model.point_cloud
scan_labels = scan_model.labels
print(f'\nScan: {len(scan_pcd.points):,} Punkte')
if scan_labels is not None:
    unique, counts = np.unique(scan_labels, return_counts=True)
    label_summary = ', '.join(f'{l}:{c:,}' for l, c in zip(unique, counts))
    print(f'  Labels: {label_summary}')
scan_bbox = scan_pcd.get_axis_aligned_bounding_box()
print(f'  BBox-Extent: {scan_bbox.get_extent()}')
print(f'  Schwerpunkt: {scan_bbox.get_center()}')

In [ ]:
print(f'Hat Labels: {scan_labels is not None}')
print(f'Files im Ordner: {sorted(p.name for p in SCAN_MODEL_DIR.iterdir())}')

## 3 – Ausgangslage vor der Registrierung

Darstellung beider Punktwolken im selben Koordinatensystem. Aus der relativen Lage lässt sich die Größenordnung der durch die Registrierung zu kompensierenden Translation und Rotation abschätzen.

In [ ]:
cad_pts = np.asarray(cad_pcd.points)
scan_pts = np.asarray(scan_pcd.points)

fig = plot_clouds([
    make_scatter(cad_pts,  'CAD',  'blue', size=1.0, opacity=0.4),
    make_scatter(scan_pts, 'Scan', 'red', size=1.5, opacity=0.8),
], title='Ausgangslage – CAD (blau) und Scan (rot) im gemeinsamen Koordinatensystem')
fig.show()

## 4 – CoarsePCA-Registrierung

Der CoarsePCA-Step wird in eine `RegistrationPipeline` eingebettet (für AP2.2 später um weitere Steps wie ICP ergänzt). Der Scan dient als Source und wird ausgerichtet, das CAD-Modell als festes Target.

Der Parameter `anchor_labels=[0, 1, 2]` beschränkt die PCA-Berechnung auf Werkstück-Oberseite und Flanken; die Spalt-Region (Label 3) und Sub-Gap-Artefakte (Label 4) werden ausgeschlossen, da sie für die Lage-Bestimmung störend wirken. `evaluation_samples=5000` legt die Größe des Subsamples für das Ranking der vier Vorzeichen-Kandidaten fest.

In [ ]:
cad_normals = np.asarray(cad_pcd.normals)
top_mask = cad_normals[:, 2] > 0.5
cad_pcd_top = cad_pcd.select_by_index(np.where(top_mask)[0])
print(f'CAD voll: {len(cad_pcd.points):,} → CAD top: {len(cad_pcd_top.points):,}')

In [ ]:
step = CoarsePCA(
    anchor_labels=[0, 1, 2],
    evaluation_samples=5_000,
)
pipeline = RegistrationPipeline([step])

scan_aligned, report = pipeline.run(
    scan_pcd, cad_pcd_top,
    source_labels=scan_labels,
    target_labels=None,
)

print(report.summary())
print(f'\nFinale Transformationsmatrix:\n{report.final_transform}')

# ICP Feinregistrierung

## 5 – Ergebnis der Grob-Registrierung

Visualisierung des ausgerichteten Scans über dem CAD-Modell. Eine erfolgreiche CoarsePCA-Registrierung bringt die Hauptachsen beider Wolken näherungsweise zur Deckung; verbleibende Restfehler werden in der nachfolgenden Fein-Registrierung kompensiert.

In [ ]:
scan_aligned_pts = np.asarray(scan_aligned.points)

fig = plot_clouds([
    make_scatter(cad_pts,          'CAD',                'blue', size=1.0, opacity=0.35),
    make_scatter(scan_aligned_pts, 'Scan ausgerichtet',  'red', size=1.5, opacity=0.85),
], title='Nach CoarsePCA – ausgerichteter Scan (rot) über CAD (blau)')
fig.show()

# 5 - ICP Feinregistrierung

In [ ]:
# Pipeline mit ICPFine erweitern und erneut ausführen

pipeline_full = RegistrationPipeline([
    CoarsePCA(anchor_labels=[0, 1, 2], evaluation_samples=5_000),
    ICPFine(
        max_correspondence_distance=1.0,
        max_iteration=50,
        anchor_labels=[0, 1, 2],
    ),
])

scan_aligned, report = pipeline_full.run(
    scan_pcd, cad_pcd_top,        # cad_pcd_top wenn der Top-Crop noch aktiv ist
    source_labels=scan_labels,
    target_labels=None,
)

print(report.summary())
print(f'\nFinale Transformationsmatrix:\n{report.final_transform}')

# ICP-spezifische Details
icp_step = report.steps[1]
icp_art = icp_step.artifacts
print(f'\nICP-Statistiken:')
print(f'  Fitness:       {icp_step.fitness:.3f}  (Anteil Source-Punkte mit Korrespondenz)')
print(f'  Inlier-RMSE:   {icp_art["inlier_rmse"]:.3f} mm')
print(f'  Source-Anker:  {icp_art["anchor_count_source"]:,}')

In [ ]:
scan_aligned_pts = np.asarray(scan_aligned.points)

fig = plot_clouds([
    make_scatter(cad_pts,          'CAD',                'blue', size=1.0, opacity=0.35),
    make_scatter(scan_aligned_pts, 'Scan ausgerichtet',  'red', size=1.5, opacity=0.85),
], title='Nach CoarsePCA und ICP – ausgerichteter Scan (rot) über CAD (blau)')
fig.show()

## 6 – Quantitative Bewertung

Vergleich des mittleren und maximalen Nächste-Nachbar-Abstands zwischen Scan und CAD vor und nach der Registrierung. Die prozentuale Reduktion des mittleren Abstands dient als kompakter Indikator für die Wirksamkeit des Schritts. Eine absolute Aussage zur Ausrichtungsqualität ist erst nach der Fein-Registrierung möglich; CoarsePCA hat die Aufgabe, ICP in die typische Konvergenzregion zu bringen (Restrotation üblicherweise unter 5°).

In [ ]:
def mean_nn_dist(src_pts, target_pcd, n_samples=5000, seed=0):
    idx = np.random.default_rng(seed).choice(
        len(src_pts), min(n_samples, len(src_pts)), replace=False,
    )
    kd = o3d.geometry.KDTreeFlann(target_pcd)
    sq = np.empty(len(idx))
    for i, j in enumerate(idx):
        _, _, d2 = kd.search_knn_vector_3d(src_pts[j], 1)
        sq[i] = d2[0]
    return float(np.sqrt(sq).mean()), float(np.sqrt(sq).max())


before_mean, before_max = mean_nn_dist(scan_pts, cad_pcd)
after_mean,  after_max  = mean_nn_dist(scan_aligned_pts, cad_pcd)

print(f'Vor CoarsePCA:   mittlerer NN = {before_mean:8.3f} mm,  max = {before_max:8.3f} mm')
print(f'Nach CoarsePCA:  mittlerer NN = {after_mean:8.3f} mm,  max = {after_max:8.3f} mm')
print(f'Reduktion des mittleren Abstands: {(1 - after_mean/before_mean)*100:.1f} %')

## 7 – Detailansicht der Naht-Region

Eingeschränkte Darstellung auf die segmentierten Flanken (Label 1 und 2) sowie die räumlich korrespondierende CAD-Region. Diese Ansicht ermöglicht die visuelle Beurteilung der Ausrichtungsqualität im für AP2.2 entscheidenden Bereich, der in der Gesamtansicht durch die umgebenden Punkte überdeckt wird.

In [ ]:
flank_mask = np.isin(scan_labels, [1, 2])
scan_flank_pts = scan_aligned_pts[flank_mask]
scan_flank_labels = scan_labels[flank_mask]

flank_bbox_min = scan_flank_pts.min(axis=0) - 5
flank_bbox_max = scan_flank_pts.max(axis=0) + 5
cad_mask = np.all((cad_pts >= flank_bbox_min) & (cad_pts <= flank_bbox_max), axis=1)
cad_crop = cad_pts[cad_mask]

traces = [make_scatter(cad_crop, 'CAD (Naht-Region)', '#90A4AE', size=1.0, opacity=0.4)]
for label_id, color, name in [(1, '#1976D2', 'Flanke A'), (2, '#FB8C00', 'Flanke B')]:
    sub = scan_flank_pts[scan_flank_labels == label_id]
    traces.append(make_scatter(sub, f'Scan {name}', color, size=2.0, opacity=0.9))

fig = plot_clouds(traces, title='Naht-Region nach CoarsePCA – CAD-Crop und segmentierte Flanken')
fig.show()

## 8 – Diagnose: Kandidaten und Eigenwerte

Auflistung der Bewertungskosten für alle vier Vorzeichen-Kombinationen sowie der Eigenwerte beider Wolken.

Liegen die Kosten zweier Kandidaten dicht beieinander, ist die Vorzeichen-Wahl numerisch wenig robust und eine fehlerhafte Ausrichtung möglich. Stark unterschiedliche Eigenwerte zwischen Scan und CAD weisen auf abweichende Punktverteilungen hin, etwa durch unterschiedliche Sampling-Muster oder durch teilweise Abdeckung des Bauteils im Scan.

In [ ]:
art = report.steps[0].artifacts

print('Kandidaten-Kosten (mittlerer NN-Abstand pro Vorzeichen-Flip):')
for flip, cost in art['candidate_costs'].items():
    marker = '  ←' if tuple(eval(flip)) == art['flip_chosen'] else ''
    print(f'  flip={flip}: {cost:.3f} mm{marker}')

print('\nEigenwerte der PCA:')
print(f"  Scan:  {np.array(art['source_eigvals'])}")
print(f"  CAD:   {np.array(art['target_eigvals'])}")